In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/aishwk/gamus-subset-5gb/manifest_train.csv
/kaggle/input/datasets/aishwk/gamus-subset-5gb/manifest_test.csv
/kaggle/input/datasets/aishwk/gamus-subset-5gb/manifest_val.csv
/kaggle/input/datasets/aishwk/gamus-subset-5gb/heights/val/PHL_6153_AGL.h5
/kaggle/input/datasets/aishwk/gamus-subset-5gb/heights/val/PHL_6343_AGL.h5
/kaggle/input/datasets/aishwk/gamus-subset-5gb/heights/val/DC_30_18_AGL.h5
/kaggle/input/datasets/aishwk/gamus-subset-5gb/heights/val/PHL_6485_AGL.h5
/kaggle/input/datasets/aishwk/gamus-subset-5gb/heights/val/DC_13_34_AGL.h5
/kaggle/input/datasets/aishwk/gamus-subset-5gb/heights/val/PHL_6575_AGL.h5
/kaggle/input/datasets/aishwk/gamus-subset-5gb/heights/val/DC_43_52_AGL.h5
/kaggle/input/datasets/aishwk/gamus-subset-5gb/heights/val/DC_58_39_AGL.h5
/kaggle/input/datasets/aishwk/gamus-subset-5gb/heights/val/PHL_6573_AGL.h5
/kaggle/input/datasets/aishwk/gamus-subset-5gb/heights/val/DC_35_44_AGL.h5
/kaggle/input/datasets/aishwk/gamus-subset-5gb/heights/

In [2]:
  !pip install -q \
      h5py \
      transformers \
      accelerate \
      rasterio \
      shapely \
      albumentations

In [3]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Kaggle GPU is not enabled")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [6]:
  from pathlib import Path

  for path in Path("/kaggle/input").iterdir():
      print(path)

  GAMUS_ROOT = Path("/kaggle/input/datasets/aishwk/gamus-subset-5gb")
  CODE_ROOT = Path("/kaggle/input/datasets/aishwk/gamus-subset-5gb")

  print("GAMUS:", GAMUS_ROOT)
  print("Code:", CODE_ROOT)


  for path in [
      GAMUS_ROOT / "images" / "train",
      GAMUS_ROOT / "heights" / "train",
      GAMUS_ROOT / "classes" / "train",
  ]:
      print(path, path.exists(), len(list(path.glob("*.h5"))))

/kaggle/input/datasets
GAMUS: /kaggle/input/datasets/aishwk/gamus-subset-5gb
Code: /kaggle/input/datasets/aishwk/gamus-subset-5gb
/kaggle/input/datasets/aishwk/gamus-subset-5gb/images/train True 530
/kaggle/input/datasets/aishwk/gamus-subset-5gb/heights/train True 530
/kaggle/input/datasets/aishwk/gamus-subset-5gb/classes/train True 530


In [7]:
  !cp "$(find /kaggle/input -name kaggle_gamus_train.py | head -1)" /kaggle/working/kaggle_gamus_train.py

In [8]:
!python /kaggle/working/kaggle_gamus_train.py --help

usage: kaggle_gamus_train.py [-h] --root ROOT [--output-dir OUTPUT_DIR]
                             [--model-id MODEL_ID] [--image-key IMAGE_KEY]
                             [--height-key HEIGHT_KEY] [--class-key CLASS_KEY]
                             [--image-size IMAGE_SIZE]
                             [--batch-size BATCH_SIZE] [--epochs EPOCHS]
                             [--inspect]

options:
  -h, --help            show this help message and exit
  --root ROOT
  --output-dir OUTPUT_DIR
  --model-id MODEL_ID
  --image-key IMAGE_KEY
  --height-key HEIGHT_KEY
  --class-key CLASS_KEY
  --image-size IMAGE_SIZE
  --batch-size BATCH_SIZE
  --epochs EPOCHS
  --inspect


In [12]:
  from pathlib import Path
  import h5py

  GAMUS_ROOT = Path("/kaggle/input/datasets/aishwk/gamus-subset-5gb")

  for modality in ["images", "heights", "classes"]:
      sample = next((GAMUS_ROOT / modality / "train").glob("*.h5"))

      print(f"\n--- {modality} ---")
      print(sample)

      with h5py.File(sample, "r") as file:
          def show_dataset(name, value):
              if isinstance(value, h5py.Dataset):
                  print(name, value.shape, value.dtype)

          file.visititems(show_dataset)


--- images ---
/kaggle/input/datasets/aishwk/gamus-subset-5gb/images/train/NYC_23501_IMG.h5
image (1024, 1024, 3) uint8

--- heights ---
/kaggle/input/datasets/aishwk/gamus-subset-5gb/heights/train/PHL_3230_AGL.h5
image (1024, 1024) float32

--- classes ---
/kaggle/input/datasets/aishwk/gamus-subset-5gb/classes/train/NYC_25477_CLS.h5
image (1024, 1024) uint8


In [13]:
  from pathlib import Path

  root = Path("/kaggle/input/datasets/aishwk/gamus-subset-5gb")

  for split in ["train", "val", "test"]:
      print(f"\n--- {split} ---")

      for folder in ["images", "heights", "classes"]:
          files = sorted((root / folder / split).glob("*.h5"))
          print(folder, "count:", len(files))
          print("examples:", [file.name for file in files[:5]])


--- train ---
images count: 530
examples: ['DC_02_25_RGB.h5', 'DC_03_23_RGB.h5', 'DC_03_25_RGB.h5', 'DC_03_28_RGB.h5', 'DC_05_20_RGB.h5']
heights count: 530
examples: ['DC_02_25_AGL.h5', 'DC_03_23_AGL.h5', 'DC_03_25_AGL.h5', 'DC_03_28_AGL.h5', 'DC_05_20_AGL.h5']
classes count: 530
examples: ['DC_02_25_CLS.h5', 'DC_03_23_CLS.h5', 'DC_03_25_CLS.h5', 'DC_03_28_CLS.h5', 'DC_05_20_CLS.h5']

--- val ---
images count: 51
examples: ['DC_13_28_RGB.h5', 'DC_13_34_RGB.h5', 'DC_16_28_RGB.h5', 'DC_17_37_RGB.h5', 'DC_19_42_RGB.h5']
heights count: 51
examples: ['DC_13_28_AGL.h5', 'DC_13_34_AGL.h5', 'DC_16_28_AGL.h5', 'DC_17_37_AGL.h5', 'DC_19_42_AGL.h5']
classes count: 51
examples: ['DC_13_28_CLS.h5', 'DC_13_34_CLS.h5', 'DC_16_28_CLS.h5', 'DC_17_37_CLS.h5', 'DC_19_42_CLS.h5']

--- test ---
images count: 53
examples: ['DC_11_15_RGB.h5', 'DC_20_23_RGB.h5', 'DC_21_17_RGB.h5', 'DC_26_08_RGB.h5', 'DC_26_26_RGB.h5']
heights count: 53
examples: ['DC_11_15_AGL.h5', 'DC_20_23_AGL.h5', 'DC_21_17_AGL.h5', 'DC_

In [14]:
  for split in ["train", "val", "test"]:
      counts = {
          folder: len(list((root / folder / split).glob("*.h5")))
          for folder in ["images", "heights", "classes"]
      }
      print(split, counts)

train {'images': 530, 'heights': 530, 'classes': 530}
val {'images': 51, 'heights': 51, 'classes': 51}
test {'images': 53, 'heights': 53, 'classes': 53}


In [11]:
  IMAGE_KEY = "image"
  HEIGHT_KEY = "height"
  CLASS_KEY = "classes"

  print(IMAGE_KEY, HEIGHT_KEY, CLASS_KEY)

image height classes


In [15]:
  from pathlib import Path
  import os

  source = Path("/kaggle/input/datasets/aishwk/gamus-subset-5gb")
  target = Path("/kaggle/working/gamus_paired")

  for split in ["train", "val", "test"]:
      for folder in ["images", "heights", "classes"]:
          (target / folder / split).mkdir(parents=True, exist_ok=True)

      image_files = sorted((source / "images" / split).glob("*_RGB.h5"))

      for image_file in image_files:
          stem = image_file.name.replace("_RGB.h5", "")

          height_file = source / "heights" / split / f"{stem}_AGL.h5"
          class_file = source / "classes" / split / f"{stem}_CLS.h5"

          if not height_file.exists():
              raise FileNotFoundError(height_file)

          if not class_file.exists():
              raise FileNotFoundError(class_file)

          links = {
              target / "images" / split / image_file.name: image_file,
              target / "heights" / split / image_file.name: height_file,
              target / "classes" / split / image_file.name: class_file,
          }

          for destination, original in links.items():
              if not destination.exists():
                  os.symlink(original, destination)

  print("Paired GAMUS dataset created successfully.")

Paired GAMUS dataset created successfully.


In [16]:
  for split in ["train", "val", "test"]:
      image_names = {
          file.name for file in (target / "images" / split).glob("*.h5")
      }
      height_names = {
          file.name for file in (target / "heights" / split).glob("*.h5")
      }
      class_names = {
          file.name for file in (target / "classes" / split).glob("*.h5")
      }

      print(
          split,
          len(image_names),
          image_names == height_names == class_names
      )

train 354 True
val 51 True
test 36 True


In [18]:
  !python /kaggle/working/kaggle_gamus_train.py \
      --root /kaggle/working/gamus_paired \
      --output-dir /kaggle/working/depthwizard_base \
      --image-key image \
      --height-key image \
      --class-key image \
      --model-id depth-anything/Depth-Anything-V2-Base-hf \
      --image-size 518 \
      --batch-size 1 \
      --epochs 30

^C
Traceback (most recent call last):
  File "/kaggle/working/kaggle_gamus_train.py", line 355, in <module>
    main()
  File "/kaggle/working/kaggle_gamus_train.py", line 278, in main
    from transformers import AutoModelForDepthEstimation
  File "<frozen importlib._bootstrap>", line 1412, in _handle_fromlist
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2098, in __getattr__
    module = self._get_module(self._class_to_module[name])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2286, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  F

In [24]:
  from pathlib import Path
  import shutil

  source = Path("/kaggle/input/datasets/kunalkokande/smoke-test/kaggle_gamus_train.py")
  destination = Path("/kaggle/working/kaggle_gamus_train.py")

  shutil.copy2(source, destination)

  print("Copied to:", destination)


  path = Path("/kaggle/working/kaggle_gamus_train.py")
  text = path.read_text()

  old = """                super().__init__()
                  self.base = parent.base
                  self.semantic_head = parent.semantic_head
                  self.boundary_head = parent.boundary_head
  """

  new = """                super().__init__()
                  self.torch = parent.torch
                  self.base = parent.base
                  self.semantic_head = parent.semantic_head
                  self.boundary_head = parent.boundary_head
  """

  if old in text:
      path.write_text(text.replace(old, new))
      print("Patch applied.")
  else:
      print("Patch already applied or code differs.")

Copied to: /kaggle/working/kaggle_gamus_train.py
Patch applied.


In [25]:
!grep -A8 "def __init__(self):" /kaggle/working/kaggle_gamus_train.py

            def __init__(self):
                super().__init__()
                self.torch = parent.torch
                self.base = parent.base
                self.semantic_head = parent.semantic_head
                self.boundary_head = parent.boundary_head

            def forward(self, pixel_values):
                output = self.base(pixel_values=pixel_values, output_hidden_states=True)


In [27]:
  !python /kaggle/working/kaggle_gamus_train.py \
      --root /kaggle/working/gamus_paired \
      --output-dir /kaggle/working/depthwizard_small \
      --image-key image \
      --height-key image \
      --class-key image \
      --model-id depth-anything/Depth-Anything-V2-Small-hf \
      --image-size 518 \
      --batch-size 2 \
      --epochs 30

Loading weights: 100%|█| 287/287 [00:00<00:00, 1572.48it/s, Materializing param=
/kaggle/working/kaggle_gamus_train.py:319: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
/pytorch/aten/src/ATen/native/cuda/NLLLoss2d.cu:73: nll_loss2d_forward_no_reduce_kernel: block: [28,0,0], thread: [320,0,0] Assertion `cur_target >= 0 && cur_target < n_classes` failed.
/pytorch/aten/src/ATen/native/cuda/NLLLoss2d.cu:73: nll_loss2d_forward_no_reduce_kernel: block: [28,0,0], thread: [322,0,0] Assertion `cur_target >= 0 && cur_target < n_classes` failed.
/pytorch/aten/src/ATen/native/cuda/NLLLoss2d.cu:73: nll_loss2d_forward_no_reduce_kernel: block: [28,0,0], thread: [324,0,0] Assertion `cur_target >= 0 && cur_target < n_classes` failed.
/pytorch/aten/src/ATen/native/cuda/NLLLoss2d.cu:73: nll_loss2d_forward_no_reduce_kernel: block: [28,0,0], thread: [326,0,0]

In [26]:
  !python  /kaggle/input/datasets/kunalkokande/smoke-test/kaggle_gamus_train.py \
      --root /kaggle/working/gamus_paired \
      --output-dir /kaggle/working/depthwizard_small \
      --image-key image \
      --height-key image \
      --class-key image \
      --model-id depth-anything/Depth-Anything-V2-Small-hf \
      --image-size 518 \
      --batch-size 2 \
      --epochs 30

^C
Traceback (most recent call last):
  File "/kaggle/input/datasets/kunalkokande/smoke-test/kaggle_gamus_train.py", line 355, in <module>
    main()
  File "/kaggle/input/datasets/kunalkokande/smoke-test/kaggle_gamus_train.py", line 278, in main
    from transformers import AutoModelForDepthEstimation
  File "<frozen importlib._bootstrap>", line 1412, in _handle_fromlist
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2098, in __getattr__
    module = self._get_module(self._class_to_module[name])
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/utils/import_utils.py", line 2286, in _get_module
    return importlib.import_module("." + module_name, self.__name__)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/importlib/__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
     

In [28]:
  import h5py
  import numpy as np
  from pathlib import Path

  root = Path("/kaggle/input/datasets/aishwk/gamus-subset-5gb")
  sample = next((root / "classes" / "train").glob("*.h5"))

  with h5py.File(sample, "r") as file:
      labels = np.asarray(file["image"])

  print("Unique labels:", np.unique(labels))
  print("Minimum:", labels.min())
  print("Maximum:", labels.max())

Unique labels: [0 1 2 4]
Minimum: 0
Maximum: 4


In [32]:
  from pathlib import Path

  path = Path("/kaggle/working/kaggle_gamus_train.py")
  text = path.read_text()

  old = """        valid = np.isfinite(height) & np.isin(classes, np.arange(CLASS_COUNT))
          height = np.nan_to_num(height, nan=0.0, posinf=0.0, neginf=0.0)
          boundary = boundary_mask(classes == BUILDING_CLASS)
  """

  new = """        valid = np.isfinite(height) & np.isin(classes, np.arange(CLASS_COUNT))
          height = np.nan_to_num(height, nan=0.0, posinf=0.0, neginf=0.0)

          # CrossEntropyLoss requires every target to be in 0..CLASS_COUNT-1.
          # Invalid/nodata pixels are replaced with ground and ignored by valid.
          classes = np.where(valid, classes, 0).astype(np.int64)

          boundary = boundary_mask(classes == BUILDING_CLASS)
  """
  if old not in text:
      print("Dataset patch not found or already applied.")
  else:
      path.write_text(text.replace(old, new))
      print("Dataset label patch applied.")

Dataset patch not found or already applied.


In [33]:
  path = Path("/kaggle/working/kaggle_gamus_train.py")
  text = path.read_text()

  old = """    semantic_loss = F.cross_entropy(outputs["semantic"], target["classes"], reduction="none")
      semantic_loss = (semantic_loss * valid[:, 0]).sum() / valid[:, 0].sum().clamp_min(1)
  """

  new = """    safe_classes = target["classes"].clamp(0, CLASS_COUNT - 1)
      semantic_loss = F.cross_entropy(outputs["semantic"], safe_classes, reduction="none")
      semantic_loss = (semantic_loss * valid[:, 0]).sum() / valid[:, 0].sum().clamp_min(1)
  """

  if old not in text:
      print("Loss patch not found or already applied.")
  else:
      path.write_text(text.replace(old, new))
      print("Loss safety patch applied.")

Loss safety patch applied.


In [34]:
  !python /kaggle/working/kaggle_gamus_train.py \
      --root /kaggle/working/gamus_paired \
      --output-dir /kaggle/working/depthwizard_smoke \
      --image-key image \
      --height-key image \
      --class-key image \
      --model-id depth-anything/Depth-Anything-V2-Small-hf \
      --image-size 518 \
      --batch-size 1 \
      --epochs 1

Loading weights: 100%|█| 287/287 [00:00<00:00, 1458.27it/s, Materializing param=
/kaggle/working/kaggle_gamus_train.py:325: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
/kaggle/working/kaggle_gamus_train.py:184: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  "height": torch.from_numpy(height[None]).float(),
/kaggle/working/kaggle_gamus_train.py:184: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing 

In [35]:
  from pathlib import Path

  root = Path("/kaggle/working/gamus_paired")

  for split in ["train", "val", "test"]:
      print(
          split,
          "images:", len(list((root / "images" / split).glob("*.h5"))),
          "heights:", len(list((root / "heights" / split).glob("*.h5"))),
          "classes:", len(list((root / "classes" / split).glob("*.h5"))),
      )

train images: 354 heights: 354 classes: 354
val images: 51 heights: 51 classes: 51
test images: 36 heights: 36 classes: 36


In [37]:
  from pathlib import Path
  import shutil
  import os

  source = Path("/kaggle/input/datasets/aishwk/gamus-subset-5gb")
  target = Path("/kaggle/working/gamus_paired")

  # Remove the incomplete paired directory
  if target.exists():
      shutil.rmtree(target)

  for split in ["train", "val", "test"]:
      for folder in ["images", "heights", "classes"]:
          (target / folder / split).mkdir(parents=True, exist_ok=True)

      image_files = sorted((source / "images" / split).glob("*.h5"))

      for image_file in image_files:
          filename = image_file.name

          if filename.endswith("_RGB.h5"):
              stem = filename[:-len("_RGB.h5")]
          elif filename.endswith("_IMG.h5"):
              stem = filename[:-len("_IMG.h5")]
          else:
              print("Unknown RGB filename:", filename)
              continue

          height_file = source / "heights" / split / f"{stem}_AGL.h5"
          class_file = source / "classes" / split / f"{stem}_CLS.h5"

          if not height_file.exists():
              raise FileNotFoundError(f"Missing height file: {height_file}")

          if not class_file.exists():
              raise FileNotFoundError(f"Missing class file: {class_file}")

          # Standardize all linked filenames to *_RGB.h5
          output_name = f"{stem}_RGB.h5"

          links = {
              target / "images" / split / output_name: image_file,
              target / "heights" / split / output_name: height_file,
              target / "classes" / split / output_name: class_file,
          }

          for destination, original in links.items():
              os.symlink(original, destination)

  print("Rebuilt paired dataset.")

  # Verify the counts:

  for split in ["train", "val", "test"]:
      counts = {
          folder: len(list((target / folder / split).glob("*.h5")))
          for folder in ["images", "heights", "classes"]
      }
      print(split, counts)

Rebuilt paired dataset.
train {'images': 530, 'heights': 530, 'classes': 530}
val {'images': 51, 'heights': 51, 'classes': 51}
test {'images': 53, 'heights': 53, 'classes': 53}


In [38]:
  !python /kaggle/working/kaggle_gamus_train.py \
      --root /kaggle/working/gamus_paired \
      --output-dir /kaggle/working/depthwizard_smoke \
      --image-key image \
      --height-key image \
      --class-key image \
      --model-id depth-anything/Depth-Anything-V2-Small-hf \
      --image-size 518 \
      --batch-size 1 \
      --epochs 1

Loading weights: 100%|█| 287/287 [00:00<00:00, 1366.40it/s, Materializing param=
/kaggle/working/kaggle_gamus_train.py:325: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
/kaggle/working/kaggle_gamus_train.py:184: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  "height": torch.from_numpy(height[None]).float(),
/kaggle/working/kaggle_gamus_train.py:184: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing 

In [40]:
  !python /kaggle/working/kaggle_gamus_train.py \
      --root /kaggle/working/gamus_paired \
      --output-dir /kaggle/working/kutta \
      --image-key image \
      --height-key image \
      --class-key image \
      --model-id depth-anything/Depth-Anything-V2-Small-hf \
      --image-size 518 \
      --batch-size 2 \
      --epochs 30

Loading weights: 100%|█| 287/287 [00:00<00:00, 1418.61it/s, Materializing param=
/kaggle/working/kaggle_gamus_train.py:325: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
/kaggle/working/kaggle_gamus_train.py:184: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  "height": torch.from_numpy(height[None]).float(),
/kaggle/working/kaggle_gamus_train.py:184: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing 

In [41]:
  !python /kaggle/working/kaggle_gamus_train.py \
      --root /kaggle/working/gamus_paired \
      --output-dir /kaggle/working/billi \
      --image-key image \
      --height-key image \
      --class-key image \
      --model-id depth-anything/Depth-Anything-V2-Base-hf \
      --image-size 518 \
      --batch-size 1 \
      --epochs 30

config.json: 100%|█████████████████████████████| 953/953 [00:00<00:00, 5.48MB/s]
model.safetensors: 100%|██████████████████████| 390M/390M [00:02<00:00, 138MB/s]
Loading weights: 100%|█| 287/287 [00:00<00:00, 1431.00it/s, Materializing param=
/kaggle/working/kaggle_gamus_train.py:325: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
/kaggle/working/kaggle_gamus_train.py:184: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  "height": torch.from_numpy(height[None]).float(),
/ka

In [42]:
  import h5py
  import numpy as np
  from pathlib import Path

  root = Path("/kaggle/working/gamus_paired")
  histogram = np.zeros(20, dtype=np.int64)

  for path in (root / "classes" / "train").glob("*.h5"):
      with h5py.File(path, "r") as file:
          labels = np.asarray(file["image"]).astype(np.int64)

      values, counts = np.unique(labels, return_counts=True)
      print(path.name, dict(zip(values.tolist(), counts.tolist())))
      for value, count in zip(values, counts):
          if 0 <= value < len(histogram):
              histogram[value] += count

  print("TOTAL CLASS HISTOGRAM:")
  for class_id, count in enumerate(histogram):
      if count:
          print(class_id, count)

PHL_2009_RGB.h5 {1: 153853, 2: 261869, 3: 291345, 5: 142991, 6: 198518}
PHL_1951_RGB.h5 {1: 321758, 2: 349461, 3: 152308, 5: 142385, 6: 82664}
NYC_23510_RGB.h5 {0: 71791, 2: 576810, 3: 92128, 5: 82559, 6: 225288}
DC_55_44_RGB.h5 {0: 171, 1: 162392, 2: 98137, 3: 328795, 5: 275334, 6: 183747}
NYC_30237_RGB.h5 {0: 24016, 2: 65890, 5: 156983, 6: 801687}
DC_44_52_RGB.h5 {0: 99, 1: 122461, 2: 132578, 3: 330819, 5: 346833, 6: 115786}
NYC_24800_RGB.h5 {0: 324402, 2: 63286, 3: 230349, 5: 204511, 6: 226028}
PHL_3363_RGB.h5 {1: 216379, 2: 223375, 3: 38541, 5: 401849, 6: 168432}
NYC_26702_RGB.h5 {0: 281586, 2: 187946, 3: 281313, 5: 90515, 6: 207216}
PHL_1464_RGB.h5 {1: 262823, 2: 91843, 3: 396705, 5: 215459, 6: 81746}
PHL_5687_RGB.h5 {1: 184711, 2: 250987, 3: 326462, 5: 156860, 6: 129556}
NYC_26703_RGB.h5 {0: 206074, 2: 207321, 3: 165380, 5: 105562, 6: 364239}
DC_41_30_RGB.h5 {0: 135, 1: 52426, 2: 33127, 3: 633405, 5: 291753, 6: 37730}
PHL_3009_RGB.h5 {1: 32118, 2: 366271, 6: 650187}
NYC_28926_RGB

In [43]:
  from pathlib import Path

  path = Path("/kaggle/working/kaggle_gamus_train.py")
  text = path.read_text()

  text = text.replace(
      'CLASS_NAMES = ("ground", "low_vegetation", "building", "water", "road", "tree")',
      'CLASS_NAMES = ("others", "ground", "low_vegetation", "building", "water", "road", "tree")'
  )

  text = text.replace(
      "BUILDING_CLASS = 2",
      "BUILDING_CLASS = 3"
  )

  path.write_text(text)

  print("Updated to seven GAMUS classes.")

Updated to seven GAMUS classes.


In [44]:
  !grep -n "CLASS_NAMES\|BUILDING_CLASS\|CLASS_COUNT" /kaggle/working/kaggle_gamus_train.py

33:CLASS_NAMES = ("others", "ground", "low_vegetation", "building", "water", "road", "tree")
34:BUILDING_CLASS = 3
35:CLASS_COUNT = len(CLASS_NAMES)
156:        valid = np.isfinite(height) & np.isin(classes, np.arange(CLASS_COUNT))
159:        # CrossEntropyLoss requires every target to be in 0..CLASS_COUNT-1.
163:        boundary = boundary_mask(classes == BUILDING_CLASS)
199:        self.semantic_head = nn.Sequential(nn.Conv2d(channels, 256, 3, padding=1), nn.GELU(), nn.Conv2d(256, CLASS_COUNT, 1))
236:    safe_classes = target["classes"].clamp(0, CLASS_COUNT - 1)
341:                "class_names": CLASS_NAMES,
357:    (args.output_dir / "model_config.json").write_text(json.dumps({"model_id": args.model_id, "image_size": args.image_size, "class_names": CLASS_NAMES, "h5_keys": checkpoint["h5_keys"]}, indent=2))


In [45]:
  !python /kaggle/working/kaggle_gamus_train.py \
      --root /kaggle/working/gamus_paired \
      --output-dir /kaggle/working/dinosaur \
      --image-key image \
      --height-key image \
      --class-key image \
      --model-id depth-anything/Depth-Anything-V2-Base-hf \
      --image-size 518 \
      --batch-size 1 \
      --epochs 30

Loading weights: 100%|█| 287/287 [00:00<00:00, 1390.18it/s, Materializing param=
/kaggle/working/kaggle_gamus_train.py:325: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda")
/kaggle/working/kaggle_gamus_train.py:184: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  "height": torch.from_numpy(height[None]).float(),
/kaggle/working/kaggle_gamus_train.py:184: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing 